<a href="https://colab.research.google.com/github/PANshian/PSA-UDL-notebook/blob/main/Notebooks/Chap13/13_4_Graph_Attention_Networks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Notebook 13.4: Graph attention networks**

This notebook builds a graph attention mechanism from scratch, as discussed in section 13.8.6 of the book and illustrated in figure 13.12c

Work through the cells below, running each cell in turn. In various places you will see the words "TODO". Follow the instructions at these places and make predictions about what is going to happen or write code to complete the functions.

Contact me at udlbookmail@gmail.com if you find any mistakes or have any suggestions.



In [1]:
import numpy as np
import matplotlib.pyplot as plt

The self-attention mechanism maps $N$ inputs $\mathbf{x}_{n}\in\mathbb{R}^{D}$ and returns $N$ outputs $\mathbf{x}'_{n}\in \mathbb{R}^{D}$.  



In [2]:
# Set seed so we get the same random numbers
np.random.seed(1)
# Number of nodes in the graph
N = 8
# Number of dimensions of each input
D = 4

# Define a graph
A = np.array([[0,1,0,1,0,0,0,0],
              [1,0,1,1,1,0,0,0],
              [0,1,0,0,1,0,0,0],
              [1,1,0,0,1,0,0,0],
              [0,1,1,1,0,1,0,1],
              [0,0,0,0,1,0,1,1],
              [0,0,0,0,0,1,0,0],
              [0,0,0,0,1,1,0,0]]);
print(A)

# Let's also define some random data
X = np.random.normal(size=(D,N))

[[0 1 0 1 0 0 0 0]
 [1 0 1 1 1 0 0 0]
 [0 1 0 0 1 0 0 0]
 [1 1 0 0 1 0 0 0]
 [0 1 1 1 0 1 0 1]
 [0 0 0 0 1 0 1 1]
 [0 0 0 0 0 1 0 0]
 [0 0 0 0 1 1 0 0]]


We'll also need the weights and biases for the keys, queries, and values (equations 12.2 and 12.4)

In [3]:
# Choose random values for the parameters
omega = np.random.normal(size=(D,D))
beta = np.random.normal(size=(D,1))
phi = np.random.normal(size=(2*D,1))

We'll need a softmax operation that operates on the columns of the matrix and a ReLU function as well

In [4]:
# Define softmax operation that works independently on each column
def softmax_cols(data_in):
  # Exponentiate all of the values
  exp_values = np.exp(data_in) ;
  # Sum over columns
  denom = np.sum(exp_values, axis = 0);
  # Replicate denominator to N rows
  denom = np.matmul(np.ones((data_in.shape[0],1)), denom[np.newaxis,:])
  # Compute softmax
  softmax = exp_values / denom
  # return the answer
  return softmax


# Define the Rectified Linear Unit (ReLU) function
def ReLU(preactivation):
  activation = preactivation.clip(0.0)
  return activation


In [5]:
# Now let's compute self attention in matrix form
def graph_attention(X,omega, beta, phi, A):
    # 从输入X获取节点数N（X形状为(D, N)）
    N = X.shape[1]

    # TODO -- Write this function (see figure 13.12c)
    # 1. Compute X_prime：对输入X做线性变换 + 偏置
    X_prime = omega @ X + beta  # 形状：(D, N)，与输入X同维度

    # 2. Compute S：计算注意力分数矩阵（所有节点对的注意力得分）
    # 构造所有节点对的拼接特征：X'_i 和 X'_j 拼接，形状(2D, N, N)
    X_i = X_prime[:, :, np.newaxis]  # 扩展为(D, N, 1)，代表每个节点i的特征
    X_j = X_prime[:, np.newaxis, :]  # 扩展为(D, 1, N)，代表每个节点j的特征
    concat_features = np.concatenate([X_i, X_j], axis=0)  # 拼接后形状(2D, N, N)
    # 用phi计算注意力分数S：phi.T @ 拼接特征 → 形状(N, N)
    S = (phi.T @ concat_features).squeeze()  # squeeze后去掉多余维度，得到(N,N)的S矩阵

    # 3. Apply mask：将非邻居节点的注意力分数设为极小值（softmax后为0）
    # A+I 为邻接矩阵加自环（每个节点关注自己），mask为True的位置是无连接的节点对
    mask = (A + np.eye(N)) == 0
    S[mask] = -1e20  # 设为极小值，softmax后这些位置概率为0

    # 4. Run softmax：计算最终注意力权重
    attention = softmax_cols(S)  # 按列做softmax，得到(N,N)的注意力矩阵

    # 5. Postmultiply X' by attention：用注意力权重聚合特征
    output_before_relu = X_prime @ attention  # 形状(D, N)，与输入X同维度

    # 6. Apply ReLU：激活函数
    output = ReLU(output_before_relu)  # 形状(D, N)，最终输出

    # 替换原错误的占位代码
    # output = np.ones_like(X) ;

    return output;

In [7]:
 # Now let's compute self attention in matrix form
def graph_attention(X,omega, beta, phi, A):

  # TODO -- Write this function (see figure 13.12c)
  # 1. Compute X_prime
  # 2. Compute S
  # 3. To apply the mask, set S to a very large negative number (e.g. -1e20) everywhere where A+I is zero
  # 4. Run the softmax function to compute the attention values
  # 5. Postmultiply X' by the attention values
  # 6. Apply the ReLU function
  # Replace this line:
  output = np.ones_like(X) ;

  return output;

In [8]:
# Test out the graph attention mechanism
np.set_printoptions(precision=3)
output = graph_attention(X, omega, beta, phi, A);
print("Correct answer is:")
print("[[0.    0.028 0.37  0.    0.97  0.    0.    0.698]")
print(" [0.    0.    0.    0.    1.184 0.    2.654 0.  ]")
print(" [1.13  0.564 0.    1.298 0.268 0.    0.    0.779]")
print(" [0.825 0.    0.    1.175 0.    0.    0.    0.  ]]]")


print("Your answer is:")
print(output)

Correct answer is:
[[0.    0.028 0.37  0.    0.97  0.    0.    0.698]
 [0.    0.    0.    0.    1.184 0.    2.654 0.  ]
 [1.13  0.564 0.    1.298 0.268 0.    0.    0.779]
 [0.825 0.    0.    1.175 0.    0.    0.    0.  ]]]
Your answer is:
[[1. 1. 1. 1. 1. 1. 1. 1.]
 [1. 1. 1. 1. 1. 1. 1. 1.]
 [1. 1. 1. 1. 1. 1. 1. 1.]
 [1. 1. 1. 1. 1. 1. 1. 1.]]


TODO -- Try to construct a dot-product self-attention mechanism as in practical 12.1 that respects the geometry of the graph and has zero attention between non-neighboring nodes by combining figures 13.12a and 13.12b.
